In [209]:
import numpy as np
import heapq
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skewnorm

# ─────────────

In [210]:

# PARAMETERS
# ─────────────────────────────────────────

SIM_TIME             = 45 * 24        # 1080 hours = 45 days
DRIVER_RATE          = 4.74           # corrected from real data
RIDER_RATE           = 34.6           # corrected from real data
PATIENCE_RATE        = 5
SPEED                = 20             # mph
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05

In [211]:
# SIM_TIME             = 45 * 24        # 1080 hours = 45 days
# DRIVER_RATE          = 3           # corrected from real data
# RIDER_RATE           = 30         # corrected from real data
# PATIENCE_RATE        = 5
# SPEED                = 20             # mph
# AREA_SIZE            = 20
# BASE_FARE            = 3
# FARE_PER_MILE        = 2
# PETROL_COST_PER_MILE = 0.20
# CVAR_ALPHA           = 0.05


In [212]:

# ─────────────────────────────────────────
# HUB PARAMETERS
# Hubs placed at dense pickup areas
# identified from real BoxCar data
# ─────────────────────────────────────────

IDLE_TIMEOUT = 0.75    # hours idle before relocating to hub (30 mins)

# Single hub placed at the densest pickup location from real data
HUBS = [
    np.array([9.79, 12.79])   # dense pickup cluster     # secondary cluster
    # dropoff-heavy area
]   # main pickup cluster (top-left)
# HUB2 = np.array([11.79, 11.79]) 


In [213]:
def driver_initial_location():
    """
    Symmetric Normal — minimal skew in driver initial location data
    Centre: (9.97, 11.51), Std: (4.36, 4.34)
    """
    x = np.clip(np.random.normal(9.97,  4.36), 0, AREA_SIZE)
    y = np.clip(np.random.normal(11.51, 4.34), 0, AREA_SIZE)
    return np.array([x, y])

def rider_pickup_location():
    """
    X: right-skewed  skewnorm(a= 1.80, loc= 4.18, scale=5.97)
    Y: left-skewed   skewnorm(a=-2.49, loc=17.05, scale=6.31)
    Produces left-centre, upper density matching real pickup data
    """
    x = np.clip(skewnorm.rvs(a= 1.80, loc= 4.18, scale=5.97), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-2.49, loc=17.05, scale=6.31), 0, AREA_SIZE)
    return np.array([x, y])

def rider_dropoff_location():
    """
    X: left-skewed         skewnorm(a= -1.65, loc=15.51, scale=6.24)
    Y: strongly left-skewed skewnorm(a=-14.26, loc=19.50, scale=7.50)
    Produces top-right density matching real dropoff data
    """
    x = np.clip(skewnorm.rvs(a= -1.65, loc=15.51, scale=6.24), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-14.26, loc=19.50, scale=7.50), 0, AREA_SIZE)
    return np.array([x, y])
    

In [214]:

# # ─────────────────────────────────────────
# # LOCATION FUNCTIONS
# # (Corrected: all follow clipped Normal,
# #  not uniform as BoxCar assumed)
# # ─────────────────────────────────────────

# def driver_initial_location():
#     """
#     Symmetric Normal — minimal skew in driver initial location data
#     Centre: (9.97, 11.51), Std: (4.36, 4.34)
#     """
#     x = np.random.uniform(0, AREA_SIZE)
#     y = np.random.uniform(0, AREA_SIZE)
#     return np.array([x, y])

# def rider_pickup_location():
#     """
#     X: right-skewed  skewnorm(a= 1.80, loc= 4.18, scale=5.97)
#     Y: left-skewed   skewnorm(a=-2.49, loc=17.05, scale=6.31)
#     Produces left-centre, upper density matching real pickup data
#     """
#     x = np.random.uniform(0, AREA_SIZE)
#     y = np.random.uniform(0, AREA_SIZE)
#     return np.array([x, y])

# def rider_dropoff_location():
#     """
#     X: left-skewed         skewnorm(a= -1.65, loc=15.51, scale=6.24)
#     Y: strongly left-skewed skewnorm(a=-14.26, loc=19.50, scale=7.50)
#     Produces top-right density matching real dropoff data
#     """
#     x = np.random.uniform(0, AREA_SIZE)
#     y = np.random.uniform(0, AREA_SIZE)
#     return np.array([x, y])

In [215]:

def nearest_hub(location):
    return min(HUBS, key=lambda h: np.linalg.norm(location - h))
# ─────────────────────────────────────────
# DRIVER CLASS
# ─────────────────────────────────────────

class Driver:
    def __init__(self, driver_id, location, online_until):
        self.id              = driver_id
        self.location        = location
        self.online_until    = online_until
        self.earnings        = 0
        self.status          = "idle"   # idle | busy | relocating
        self.busy_time       = 0
        self.relocation_cost = 0        # total petrol spent relocating to hub


In [216]:

# ─────────────────────────────────────────
# RIDER CLASS
# ─────────────────────────────────────────

class Rider:
    def __init__(self, rider_id, origin, destination, arrival_time, abandon_time):
        self.id           = rider_id
        self.origin       = origin
        self.destination  = destination
        self.arrival_time = arrival_time
        self.abandon_time = abandon_time
        self.matched      = False


In [217]:

# ─────────────────────────────────────────
# SIMULATION CLASS
# ─────────────────────────────────────────

class Simulation:

    def __init__(self, use_hubs=False):
        self.use_hubs = use_hubs          # toggle hub policy on/off

        self.current_time    = 0
        self.event_list      = []

        self.idle_drivers     = {}
        self.busy_drivers     = {}
        self.relocating_drivers = {}      # drivers currently driving to hub
        self.waiting_riders   = {}

        self.driver_counter  = 0
        self.rider_counter   = 0

        self.total_riders     = 0
        self.abandoned_riders = 0
        self.total_wait_time  = 0

        self.driver_log = []
        self.rider_log  = []

    # ─────────────────────────────────────
    # HELPERS
    # ─────────────────────────────────────

    def exp_time(self, rate):
        return np.random.exponential(1 / rate)

    def distance(self, a, b):
        return np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

    def travel_time(self, dist):
        mu = dist / SPEED
        return np.random.uniform(0.8 * mu, 1.2 * mu)

    def schedule_event(self, time, event_type, data=None):
        heapq.heappush(self.event_list, (time, event_type, data))

    # ─────────────────────────────────────
    # MATCHING
    # ─────────────────────────────────────

    def try_match(self):
        if not self.idle_drivers or not self.waiting_riders:
            return

        driver_id = next(iter(self.idle_drivers))
        driver    = self.idle_drivers[driver_id]

        rider_id = min(
            self.waiting_riders,
            key=lambda r: self.distance(driver.location,
                                        self.waiting_riders[r].origin)
        )

        rider         = self.waiting_riders[rider_id]
        rider.matched = True

        wait_time             = self.current_time - rider.arrival_time
        self.total_wait_time += wait_time

        pickup_dist     = self.distance(driver.location, rider.origin)
        trip_dist       = self.distance(rider.origin, rider.destination)
        pickup_time     = self.travel_time(pickup_dist)
        trip_time       = self.travel_time(trip_dist)
        total_trip_time = pickup_time + trip_time

        payment  = BASE_FARE + FARE_PER_MILE * trip_dist
        petrol   = PETROL_COST_PER_MILE * (pickup_dist + trip_dist)
        earnings = payment - petrol

        driver.status    = "busy"
        driver.busy_time += total_trip_time

        self.busy_drivers[driver_id] = driver
        del self.idle_drivers[driver_id]
        del self.waiting_riders[rider_id]

        dropoff_time = self.current_time + total_trip_time
        self.schedule_event(dropoff_time, "DROPOFF",
                            (driver_id, rider.destination, earnings))

    # ─────────────────────────────────────
    # EVENT HANDLERS
    # ─────────────────────────────────────

    def handle_driver_arrival(self):
        self.driver_counter += 1
        driver_id = self.driver_counter

        location        = driver_initial_location()
        online_duration = np.random.uniform(6, 8)
        online_until    = self.current_time + online_duration

        driver = Driver(driver_id, location, online_until)
        self.idle_drivers[driver_id] = driver

        self.driver_log.append({
            "driver_id"      : driver_id,
            "arrival_time"   : self.current_time,
            "online_until"   : online_until,
            "initial_x"      : location[0],
            "initial_y"      : location[1],
            "earnings"       : 0,
            "busy_time"      : 0,
            "relocation_cost": 0,
            "n_relocations"  : 0
        })

        self.schedule_event(online_until, "DRIVER_OFFLINE", driver_id)
        self.schedule_event(self.current_time + self.exp_time(DRIVER_RATE),
                            "DRIVER_ARRIVAL")

        # Schedule idle timeout if hubs are enabled
        if self.use_hubs:
            self.schedule_event(self.current_time + IDLE_TIMEOUT,
                                "DRIVER_IDLE_TIMEOUT", driver_id)

        self.try_match()

    def handle_rider_arrival(self):
        self.rider_counter += 1
        rider_id           = self.rider_counter
        self.total_riders += 1

        origin       = rider_pickup_location()
        destination  = rider_dropoff_location()
        abandon_time = self.current_time + self.exp_time(PATIENCE_RATE)

        rider = Rider(rider_id, origin, destination,
                      self.current_time, abandon_time)
        self.waiting_riders[rider_id] = rider

        self.rider_log.append({
            "rider_id"    : rider_id,
            "arrival_time": self.current_time,
            "origin_x"    : origin[0],
            "origin_y"    : origin[1],
            "dest_x"      : destination[0],
            "dest_y"      : destination[1]
        })

        self.schedule_event(abandon_time, "RIDER_ABANDON", rider_id)
        self.schedule_event(self.current_time + self.exp_time(RIDER_RATE),
                            "RIDER_ARRIVAL")
        self.try_match()

    def handle_dropoff(self, data):
        driver_id, new_location, earnings = data

        driver           = self.busy_drivers[driver_id]
        driver.earnings += earnings
        driver.location  = new_location
        driver.status    = "idle"

        for d in self.driver_log:
            if d["driver_id"] == driver_id:
                d["earnings"]  += earnings
                d["busy_time"]  = driver.busy_time
                break

        del self.busy_drivers[driver_id]

        if self.current_time < driver.online_until:
            self.idle_drivers[driver_id] = driver

            # After every dropoff, schedule idle timeout —
            # if unmatched for 30 mins at new location → return to hub
            if self.use_hubs:
                self.schedule_event(self.current_time + IDLE_TIMEOUT,
                                    "DRIVER_IDLE_TIMEOUT", driver_id)

        self.try_match()

    def handle_idle_timeout(self, driver_id):
        """
        Fires IDLE_TIMEOUT hours after a driver becomes idle.
        If still idle → always relocate back to hub, regardless of
        how many times they have already been there.
        This means: after every trip, if unmatched for 30 mins → return to hub.
        """
        if driver_id not in self.idle_drivers:
            return

        driver = self.idle_drivers[driver_id]

        if self.current_time >= driver.online_until:
            return

        hub = nearest_hub(driver.location)
        reloc_dist = np.linalg.norm(driver.location - hub)  
        # Already at hub (within 0.5 miles) — no need to move
        if reloc_dist < 0.5:
            return

        reloc_time        = self.travel_time(reloc_dist)
        reloc_petrol_cost = PETROL_COST_PER_MILE * reloc_dist

        # Driver pays own petrol to relocate (unpaid journey)
        driver.earnings        -= reloc_petrol_cost
        driver.relocation_cost += reloc_petrol_cost
        driver.status           = "relocating"

        for d in self.driver_log:
            if d["driver_id"] == driver_id:
                d["earnings"]        -= reloc_petrol_cost
                d["relocation_cost"] += reloc_petrol_cost
                d["n_relocations"]   += 1
                break

        self.relocating_drivers[driver_id] = driver
        del self.idle_drivers[driver_id]

        arrive_hub_time = self.current_time + reloc_time
        self.schedule_event(arrive_hub_time, "DRIVER_AT_HUB",(driver_id, hub.copy()))

    def handle_driver_at_hub(self, data):
        """
        Driver has arrived at hub — idle and available for matching.
        Schedules another idle timeout so if still unmatched after
        30 mins at hub AND then does a trip and comes back elsewhere,
        the cycle repeats: idle → timeout → return to hub.
        """
        driver_id, hub_location = data

        if driver_id not in self.relocating_drivers:
            return

        driver          = self.relocating_drivers[driver_id]
        driver.location = hub_location
        driver.status   = "idle"

        del self.relocating_drivers[driver_id]

        if self.current_time < driver.online_until:
            self.idle_drivers[driver_id] = driver

            # Schedule idle timeout again — if still unmatched after
            # another 30 mins at hub, they stay (already at hub so
            # reloc_dist < 0.5 check will catch it and do nothing)
            self.schedule_event(self.current_time + IDLE_TIMEOUT,
                                "DRIVER_IDLE_TIMEOUT", driver_id)

        self.try_match()

    # ─────────────────────────────────────
    # MAIN RUN
    # ─────────────────────────────────────

    def run(self):
        self.schedule_event(0, "DRIVER_ARRIVAL")
        self.schedule_event(0, "RIDER_ARRIVAL")

        while self.event_list and self.current_time < SIM_TIME:
            self.current_time, event_type, data = heapq.heappop(self.event_list)

            if event_type == "DRIVER_ARRIVAL":
                self.handle_driver_arrival()

            elif event_type == "RIDER_ARRIVAL":
                self.handle_rider_arrival()

            elif event_type == "DROPOFF":
                self.handle_dropoff(data)

            elif event_type == "RIDER_ABANDON":
                if data in self.waiting_riders:
                    self.abandoned_riders += 1
                    del self.waiting_riders[data]

            elif event_type == "DRIVER_OFFLINE":
                if data in self.idle_drivers:
                    del self.idle_drivers[data]
                if data in self.relocating_drivers:
                    del self.relocating_drivers[data]

            elif event_type == "DRIVER_IDLE_TIMEOUT":
                if self.use_hubs:
                    self.handle_idle_timeout(data)

            elif event_type == "DRIVER_AT_HUB":
                self.handle_driver_at_hub(data)

        return self.collect_results()

    # ─────────────────────────────────────
    # COLLECT RESULTS
    # ─────────────────────────────────────

    def collect_results(self):
        """Return all metrics as a dict for comparison."""
        matched_riders   = self.total_riders - self.abandoned_riders
        abandonment_rate = self.abandoned_riders / self.total_riders
        avg_wait         = self.total_wait_time / matched_riders if matched_riders > 0 else 0

        driver_df = pd.DataFrame(self.driver_log)
        driver_df["online_duration"]   = driver_df["online_until"] - driver_df["arrival_time"]
        driver_df["earnings_per_hour"] = driver_df["earnings"] / driver_df["online_duration"]
        driver_df["busy_time_capped"]  = driver_df[["busy_time","online_duration"]].min(axis=1)
        driver_df["idle_time"]         = driver_df["online_duration"] - driver_df["busy_time_capped"]

        profits     = driver_df["earnings"].values
        N           = len(profits)
        mean_profit = profits.mean()
        var_profit  = np.sum((profits - mean_profit)**2) / (N - 1) if N > 1 else 0
        std_profit  = np.sqrt(var_profit)
        fairness_cv = std_profit / mean_profit if mean_profit > 0 else 0

        profits_sorted = np.sort(profits)
        cutoff         = max(1, int(np.floor(CVAR_ALPHA * N)))
        cvar           = np.mean(profits_sorted[:cutoff])
        cvar_median    = np.mean(profits_sorted[:max(1, int(np.floor(0.5 * N)))])
        delta_cvar     = cvar - cvar_median

        total_reloc_cost = driver_df["relocation_cost"].sum()
        avg_relocations  = driver_df["n_relocations"].mean()

        return {
            # Rider metrics
            "total_riders"      : self.total_riders,
            "matched_riders"    : matched_riders,
            "abandoned_riders"  : self.abandoned_riders,
            "abandonment_rate"  : abandonment_rate,
            "avg_wait"          : avg_wait,
            # Driver metrics
            "total_drivers"     : N,
            "total_online_hrs"  : driver_df["online_duration"].sum(),
            "total_earnings"    : driver_df["earnings"].sum(),
            "avg_profit_per_hr" : driver_df["earnings"].sum() / driver_df["online_duration"].sum(),
            "avg_earnings"      : mean_profit,
            "avg_idle_time"     : driver_df["idle_time"].mean(),
            # Fairness
            "variance"          : var_profit,
            "std_dev"           : std_profit,
            "fairness_cv"       : fairness_cv,
            "cvar"              : cvar,
            "delta_cvar"        : delta_cvar,
            # Hub-specific
            "total_reloc_cost"  : total_reloc_cost,
            "avg_relocations"   : avg_relocations,
            # dataframe for plots
            "_driver_df"        : driver_df,
            "_rider_df"         : pd.DataFrame(self.rider_log),
        }

    # ─────────────────────────────────────
    # REPORT
    # ─────────────────────────────────────

    def report(self, results, label=""):
        r = results
        print(f"\n{'='*50}")
        print(f"  SIMULATION REPORT  {label}")
        print(f"{'='*50}")

        print("\n--- 1.1 RIDERS' SATISFACTION ---")
        print(f"  Total Riders Arrived     : {r['total_riders']}")
        print(f"  Matched Riders           : {r['matched_riders']}")
        print(f"  Abandoned Riders         : {r['abandoned_riders']}")
        print(f"  Abandonment Rate         : {round(r['abandonment_rate'], 4)}")
        print(f"  Average Waiting Time     : {round(r['avg_wait'], 4)} hrs")

        print("\n--- 1.2 DRIVERS' SATISFACTION ---")
        print(f"  Total Drivers            : {r['total_drivers']}")
        print(f"  Total Online Hours       : {round(r['total_online_hrs'], 2)}")
        print(f"  Total Earnings           : £{round(r['total_earnings'], 2)}")

        print("\n  [1.2.1] Average Profit")
        print(f"  Avg Profit / Online Hour : £{round(r['avg_profit_per_hr'], 4)}")
        print(f"  Avg Earnings / Driver    : £{round(r['avg_earnings'], 4)}")
        print(f"  Average Idle Time        : {round(r['avg_idle_time'], 4)} hrs")

        print("\n  [1.2.2] Fairness")
        print(f"  Variance of Profits      : {round(r['variance'], 4)}")
        print(f"  Std Dev of Profits       : {round(r['std_dev'], 4)}")
        print(f"  Fairness CV              : {round(r['fairness_cv'], 4)}  (lower = fairer)")
        print(f"  CVaR (5th pct)           : £{round(r['cvar'], 4)}")
        print(f"  Delta CVaR               : {round(r['delta_cvar'], 4)}")

        if self.use_hubs:
            print("\n  [1.2.3] Hub Repositioning Cost")
            print(f"  Total Relocation Cost    : £{round(r['total_reloc_cost'], 2)}")
            print(f"  Avg Relocations/Driver   : {round(r['avg_relocations'], 2)}")

        print(f"\n{'='*50}")


# ─────────────────────────────────────────
# RUN BOTH SIMULATIONS & COMPARE
# ─────────────────────────────────────────

np.random.seed(42)
print("Running baseline simulation (no hubs)...")
sim_base = Simulation(use_hubs=True)
results_base = sim_base.run()
sim_base.report(results_base, label="[ BASELINE — No Hubs ]")


Running baseline simulation (no hubs)...

  SIMULATION REPORT  [ BASELINE — No Hubs ]

--- 1.1 RIDERS' SATISFACTION ---
  Total Riders Arrived     : 37335
  Matched Riders           : 36150
  Abandoned Riders         : 1185
  Abandonment Rate         : 0.0317
  Average Waiting Time     : 0.0048 hrs

--- 1.2 DRIVERS' SATISFACTION ---
  Total Drivers            : 5160
  Total Online Hours       : 36135.21
  Total Earnings           : £580628.59

  [1.2.1] Average Profit
  Avg Profit / Online Hour : £16.0682
  Avg Earnings / Driver    : £112.5249
  Average Idle Time        : 1.4299 hrs

  [1.2.2] Fairness
  Variance of Profits      : 488.2127
  Std Dev of Profits       : 22.0955
  Fairness CV              : 0.1964  (lower = fairer)
  CVaR (5th pct)           : £68.2221
  Delta CVaR               : -26.7829

  [1.2.3] Hub Repositioning Cost
  Total Relocation Cost    : £4324.3
  Avg Relocations/Driver   : 0.82



In [218]:
SIM_TIME             = 45 * 24        # 1080 hours = 45 days
DRIVER_RATE          = 4.74           # corrected from real data
RIDER_RATE           = 34.6           # corrected from real data
PATIENCE_RATE        = 5
SPEED                = 20             # mph
AREA_SIZE            = 20
BASE_FARE            = 3
FARE_PER_MILE        = 2
PETROL_COST_PER_MILE = 0.20
CVAR_ALPHA           = 0.05
HUBS = [
    np.array([9.79, 12.79]),   # dense pickup cluster
    np.array([14.79, 9.79])     # secondary cluster
    # dropoff-heavy area
] 
def driver_initial_location():
    """
    Symmetric Normal — minimal skew in driver initial location data
    Centre: (9.97, 11.51), Std: (4.36, 4.34)
    """
    x = np.clip(np.random.normal(9.97,  4.36), 0, AREA_SIZE)
    y = np.clip(np.random.normal(11.51, 4.34), 0, AREA_SIZE)
    return np.array([x, y])

def rider_pickup_location():
    """
    X: right-skewed  skewnorm(a= 1.80, loc= 4.18, scale=5.97)
    Y: left-skewed   skewnorm(a=-2.49, loc=17.05, scale=6.31)
    Produces left-centre, upper density matching real pickup data
    """
    x = np.clip(skewnorm.rvs(a= 1.80, loc= 4.18, scale=5.97), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-2.49, loc=17.05, scale=6.31), 0, AREA_SIZE)
    return np.array([x, y])

def rider_dropoff_location():
    """
    X: left-skewed         skewnorm(a= -1.65, loc=15.51, scale=6.24)
    Y: strongly left-skewed skewnorm(a=-14.26, loc=19.50, scale=7.50)
    Produces top-right density matching real dropoff data
    """
    x = np.clip(skewnorm.rvs(a= -1.65, loc=15.51, scale=6.24), 0, AREA_SIZE)
    y = np.clip(skewnorm.rvs(a=-14.26, loc=19.50, scale=7.50), 0, AREA_SIZE)
    return np.array([x, y])

np.random.seed(42)
print("\nRunning hub simulation...")
sim_hub = Simulation(use_hubs=True)
results_hub = sim_hub.run()
sim_hub.report(results_hub, label="[ WITH HUB REPOSITIONING ]")



Running hub simulation...

  SIMULATION REPORT  [ WITH HUB REPOSITIONING ]

--- 1.1 RIDERS' SATISFACTION ---
  Total Riders Arrived     : 37347
  Matched Riders           : 36215
  Abandoned Riders         : 1132
  Abandonment Rate         : 0.0303
  Average Waiting Time     : 0.0046 hrs

--- 1.2 DRIVERS' SATISFACTION ---
  Total Drivers            : 5198
  Total Online Hours       : 36415.35
  Total Earnings           : £581829.94

  [1.2.1] Average Profit
  Avg Profit / Online Hour : £15.9776
  Avg Earnings / Driver    : £111.9334
  Average Idle Time        : 1.4458 hrs

  [1.2.2] Fairness
  Variance of Profits      : 456.7767
  Std Dev of Profits       : 21.3723
  Fairness CV              : 0.1909  (lower = fairer)
  CVaR (5th pct)           : £67.2429
  Delta CVaR               : -27.936

  [1.2.3] Hub Repositioning Cost
  Total Relocation Cost    : £3646.24
  Avg Relocations/Driver   : 0.78



In [219]:

# ─────────────────────────────────────────
# COMPARISON TABLE
# ─────────────────────────────────────────

b, h = results_base, results_hub

def delta(before, after, pct=False):
    d = after - before
    if pct:
        return f"{'+' if d>=0 else ''}{d/abs(before)*100:.1f}%"
    return f"{'+' if d>=0 else ''}{round(d, 4)}"

print("\n" + "="*65)
print("  BEFORE vs AFTER COMPARISON")
print("="*65)
print(f"  {'Metric':<30} {'Baseline':>12} {'With Hubs':>12} {'Change':>10}")
print("-"*65)
print(f"  {'Abandonment Rate':<30} {b['abandonment_rate']:>12.4f} {h['abandonment_rate']:>12.4f} {delta(b['abandonment_rate'], h['abandonment_rate']):>10}")
print(f"  {'Avg Wait Time (hrs)':<30} {b['avg_wait']:>12.4f} {h['avg_wait']:>12.4f} {delta(b['avg_wait'], h['avg_wait']):>10}")
print(f"  {'Matched Riders':<30} {b['matched_riders']:>12} {h['matched_riders']:>12} {delta(b['matched_riders'], h['matched_riders']):>10}")
print(f"  {'Avg Earnings/Driver (£)':<30} {b['avg_earnings']:>12.2f} {h['avg_earnings']:>12.2f} {delta(b['avg_earnings'], h['avg_earnings']):>10}")
print(f"  {'Avg Profit/Hour (£)':<30} {b['avg_profit_per_hr']:>12.4f} {h['avg_profit_per_hr']:>12.4f} {delta(b['avg_profit_per_hr'], h['avg_profit_per_hr']):>10}")
print(f"  {'Avg Idle Time (hrs)':<30} {b['avg_idle_time']:>12.4f} {h['avg_idle_time']:>12.4f} {delta(b['avg_idle_time'], h['avg_idle_time']):>10}")
print(f"  {'Fairness CV':<30} {b['fairness_cv']:>12.4f} {h['fairness_cv']:>12.4f} {delta(b['fairness_cv'], h['fairness_cv']):>10}")
print(f"  {'CVaR 5th pct (£)':<30} {b['cvar']:>12.2f} {h['cvar']:>12.2f} {delta(b['cvar'], h['cvar']):>10}")
print(f"  {'Total Relocation Cost (£)':<30} {'N/A':>12} {h['total_reloc_cost']:>12.2f} {'':>10}")
print(f"  {'Avg Relocations/Driver':<30} {'N/A':>12} {h['avg_relocations']:>12.2f} {'':>10}")
print("="*65)

# ─────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────

# # Plot 1: Side-by-side bar chart comparison
# metrics  = ['Abandonment\nRate', 'Avg Wait\nTime (hrs)',
#             'Fairness CV', 'Avg Idle\nTime (hrs)']
# baseline = [b['abandonment_rate'], b['avg_wait'],
#             b['fairness_cv'],      b['avg_idle_time']]
# hub_vals = [h['abandonment_rate'], h['avg_wait'],
#             h['fairness_cv'],      h['avg_idle_time']]

# x   = np.arange(len(metrics))
# w   = 0.35
# fig, ax = plt.subplots(figsize=(12, 6))
# bars1 = ax.bar(x - w/2, baseline, w, label='Baseline (No Hubs)',
#                color='steelblue', alpha=0.85)
# bars2 = ax.bar(x + w/2, hub_vals, w, label='With Hub Repositioning',
#                color='darkorange', alpha=0.85)
# ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
# ax.set_title('Performance Comparison: Baseline vs Hub Repositioning', fontsize=13)
# ax.set_ylabel('Value'); ax.legend()
# ax.bar_label(bars1, fmt='%.4f', padding=3, fontsize=8)
# ax.bar_label(bars2, fmt='%.4f', padding=3, fontsize=8)
# plt.tight_layout()

# plt.close()

# # Plot 2: Earnings distribution before vs after
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# df_b = results_base['_driver_df']
# df_h = results_hub['_driver_df']

# axes[0].hist(df_b['earnings'], bins=40, color='steelblue',
#              edgecolor='white', alpha=0.8)
# axes[0].axvline(df_b['earnings'].mean(), color='red', linestyle='--',
#                 label=f"Mean: £{df_b['earnings'].mean():.2f}")
# axes[0].set_title('Driver Earnings — Baseline')
# axes[0].set_xlabel('Earnings (£)'); axes[0].set_ylabel('Drivers')
# axes[0].legend()

# axes[1].hist(df_h['earnings'], bins=40, color='darkorange',
#              edgecolor='white', alpha=0.8)
# axes[1].axvline(df_h['earnings'].mean(), color='red', linestyle='--',
#                 label=f"Mean: £{df_h['earnings'].mean():.2f}")
# axes[1].set_title('Driver Earnings — With Hub Repositioning')
# axes[1].set_xlabel('Earnings (£)'); axes[1].set_ylabel('Drivers')
# axes[1].legend()

# plt.tight_layout()
# plt.close()

# # Plot 3: Hub locations on map with pickup density
# fig, ax = plt.subplots(figsize=(8, 8))
# rider_df = results_hub['_rider_df']
# ax.scatter(rider_df['origin_x'], rider_df['origin_y'],
#            s=1, alpha=0.2, color='blue', label='Rider Pickups')
# ax.scatter(HUB[0], HUB[1], s=400, marker='*',
#            color='red', zorder=5,
#            label=f'Hub ({HUB[0]:.1f}, {HUB[1]:.1f})')
# circle = plt.Circle(HUB, 2, color='red', fill=False,
#                     linestyle='--', alpha=0.5)
# ax.add_patch(circle)
# ax.set_xlim(0, AREA_SIZE); ax.set_ylim(0, AREA_SIZE)
# ax.set_xlabel('Longitude / X'); ax.set_ylabel('Latitude / Y')
# ax.set_title('Hub Locations vs Rider Pickup Density')
# ax.legend(loc='lower right')
# plt.tight_layout()
# plt.close()

# print("\nAll plots saved.")


  BEFORE vs AFTER COMPARISON
  Metric                             Baseline    With Hubs     Change
-----------------------------------------------------------------
  Abandonment Rate                     0.0317       0.0303    -0.0014
  Avg Wait Time (hrs)                  0.0048       0.0046    -0.0003
  Matched Riders                        36150        36215        +65
  Avg Earnings/Driver (£)              112.52       111.93    -0.5915
  Avg Profit/Hour (£)                 16.0682      15.9776    -0.0906
  Avg Idle Time (hrs)                  1.4299       1.4458    +0.0159
  Fairness CV                          0.1964       0.1909    -0.0054
  CVaR 5th pct (£)                      68.22        67.24    -0.9792
  Total Relocation Cost (£)               N/A      3646.24           
  Avg Relocations/Driver                  N/A         0.78           
